<a href="https://colab.research.google.com/github/signur43/Curso_DataScience/blob/Notebook/proyecto_90460_data_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Utilización del DataFrame de Reclamos Bancarios

Este DataFrame, cargado desde **`complaints_chile_banking.csv`**, es el corazón de nuestro análisis. Contiene información detallada sobre reclamos de clientes bancarios, y lo utilizaremos para:

1.  **Exploración de Datos (EDA):** Entender la distribución de los reclamos, identificar tendencias, y visualizar cómo se comportan las diferentes variables.
2.  **Preparación de Datos:** Procesar las variables categóricas (como instituciones, productos, motivos, etc.) mediante técnicas de codificación (One-Hot Encoding y Label Encoding) para que puedan ser utilizadas por los modelos de Machine Learning.
3.  **Modelado Predictivo:** Construir modelos (inicialmente de regresión lineal para predecir `tiempo_respuesta_dias`, y luego de clasificación, como Regresión Logística y Random Forest, para predecir la `satisfaccion` del cliente).
4.  **Identificación de Factores Clave:** Determinar qué variables tienen mayor influencia en el tiempo de respuesta y, crucialmente, en la satisfacción del cliente.

Nuestro objetivo final es **evaluar la satisfacción de los clientes** con sus reclamos, utilizando las variables disponibles para crear modelos que nos permitan **identificar los factores más relevantes** que influyen en dicha satisfacción.

## Carga y Primera Inspección de Datos

El primer paso en nuestro análisis es cargar el conjunto de datos de reclamos bancarios. Este DataFrame es la base de todo nuestro estudio, y es crucial realizar una primera inspección para entender su estructura, identificar tipos de datos, y verificar la presencia de valores nulos o inconsistencias.

Utilizamos la librería `pandas` para cargar el archivo CSV directamente desde una URL de GitHub, lo que asegura que el proyecto sea reproducible con los datos exactos que estamos utilizando.

In [ ]:
#dataframe para el proyecto
github_csv_url = 'https://raw.githubusercontent.com/signur43/Curso_DataScience/main/complaints_chile_banking.csv'

## Promedio de Tiempo de Respuesta por Institución

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Carga desde la URL de GitHub
df = pd.read_csv(github_csv_url, nrows=15000)

# Diagnóstico: Imprimir las columnas para verificar
print("Columnas del DataFrame:", df.columns)

# Visualización de la distribución de productos
plt.figure(figsize=(10,6))
sns.countplot(y='producto', data=df, order=df['producto'].value_counts().index)
plt.title('Distribución de Reclamos por Categoría')
plt.show()



### Conteo de Valores Únicos por Columna

Entender la cardinalidad de cada columna es fundamental. Un alto número de valores únicos en columnas categóricas puede indicar la necesidad de un tratamiento diferente o más complejo, mientras que un bajo número sugiere que son buenas candidatas para la codificación.

In [ ]:
print("Conteo de valores únicos por columna:")
display(df.nunique().sort_values(ascending=False))

### Estadísticas Descriptivas Básicas

Una vez que los datos están cargados, calculamos algunas estadísticas descriptivas básicas para las columnas numéricas como `tiempo_respuesta_dias`. Esto nos permite obtener una idea rápida de la tendencia central (media, mediana, moda) y la dispersión de estas variables.

In [ ]:
#Calcular la Media de tiempo_respuesta_dias
# Cálculo usando statistics la mediana

import statistics
media = statistics.mean(df['tiempo_respuesta_dias'])
print(f"La media es: {media}")


# Cálculo el promedio
promedio_columna = df['tiempo_respuesta_dias'].mean()
print(f"El promedio de la columna es: {promedio_columna}")

# Calcular la moda
moda = statistics.mode(df['tiempo_respuesta_dias'])
print(f"La moda es: {moda}")

### Vista Previa del DataFrame

Mostrar las primeras filas del DataFrame (`df.head()`) y sus dimensiones (`df.shape`) es fundamental para confirmar que los datos se han cargado correctamente y para tener una idea inicial de las columnas y el número de registros con los que estamos trabajando.

In [ ]:
df.head(10)

In [ ]:
df.shape

## Diccionario de Datos del DataFrame Original

Aquí se detalla cada columna del DataFrame, su tipo de dato y una breve descripción:

| Nombre de Columna       | Tipo de Dato Original | Descripción                                                                |
|:------------------------|:----------------------|:---------------------------------------------------------------------------|
| `id_reclamo`            | Object (string)       | Identificador único para cada reclamo.                                     |
| `fecha_ingreso`         | Object (string)       | Fecha en que se ingresó el reclamo.                                        |
| `institucion`           | Object (string)       | Institución financiera involucrada en el reclamo.                          |
| `producto`              | Object (string)       | Tipo de producto o servicio al que se refiere el reclamo.                  |
| `motivo_principal`      | Object (string)       | Razón principal del reclamo.                                               |
| `canal_ingreso`         | Object (string)       | Canal a través del cual se presentó el reclamo (ej. App Móvil, Sitio Web). |
| `region_usuario`        | Object (string)       | Región geográfica del usuario que presentó el reclamo.                     |
| `monto_disputado_clp`   | Integer               | Monto disputado en Pesos Chilenos (CLP).                                   |
| `estado_actual`         | Object (string)       | Estado actual del reclamo (ej. Resuelto, En proceso).                      |
| `tiempo_respuesta_dias` | Integer               | Número de días que tardó en responderse al reclamo.                        |

### Verificación de Valores Nulos

Antes de cualquier procesamiento o modelado, es esencial verificar la existencia de valores nulos en el DataFrame. La presencia de nulos puede afectar el rendimiento de los modelos, y su manejo (eliminación, imputación) es un paso crítico en la preparación de datos.

In [ ]:
# Análisis de valores nulos
print(df.isnull().sum())

### Aplicación de One-Hot Encoding

Para las columnas categóricas nominales (`institucion`), que no tienen un orden inherente, aplicamos One-Hot Encoding. Esta técnica convierte cada categoría en una nueva columna binaria (0 o 1), lo cual es ideal para evitar que el modelo interprete un orden artificial entre categorías. Se excluyen las columnas ya procesadas o que no son adecuadas para esta codificación.

## ¿Qué es la codificación (Encoding)?

La codificación es el proceso de convertir datos categóricos (datos no numéricos, como nombres de bancos, tipos de productos, etc.) en un formato numérico que los algoritmos de machine learning puedan entender y procesar. Esto es crucial porque la mayoría de los modelos de ML solo trabajan con entradas numéricas.

### Tipos comunes de codificación:

1.  **One-Hot Encoding**: Crea una nueva columna binaria (0 o 1) para cada categoría única en la columna original. Si una fila pertenece a esa categoría, la columna correspondiente tendrá un 1; de lo contrario, un 0. Es ideal para categorías nominales (sin orden intrínseco).

2.  **Label Encoding**: Asigna un número entero único a cada categoría. Por ejemplo, 'Banco A' podría ser 0, 'Banco B' 1, 'Banco C' 2, etc. Es útil para categorías ordinales (con un orden natural, como 'pequeño', 'mediano', 'grande'), pero puede introducir un orden artificial si se usa en categorías nominales.

### Columnas adecuadas para codificación:

Las siguientes columnas son de tipo `object` (cadena de texto) y representan categorías, por lo que son excelentes candidatas para la codificación:

*   `institucion`
*   `producto`
*   `motivo_principal`
*   `canal_ingreso`
*   `region_usuario`
*   `estado_actual`



In [ ]:
# Aplicar One-Hot Encoding a la columna 'institucion'
df_encoded = pd.get_dummies(df, columns=['institucion'], prefix='institucion')

# Mostrar las primeras filas del DataFrame con las nuevas columnas codificadas
print("DataFrame después de aplicar One-Hot Encoding a 'institucion':")
display(df_encoded.head())

# Mostrar las nuevas columnas creadas para 'institucion'
print("Nuevas columnas de One-Hot Encoding para 'institucion':")
print([col for col in df_encoded.columns if 'institucion_' in col])

### Aplicación de Label Encoding

Para columnas categóricas con un posible orden implícito o para reducir la dimensionalidad si hay muchas categorías, se puede utilizar Label Encoding. En este caso, aplicamos Label Encoding a `canal_ingreso`, asignando un valor numérico único a cada categoría del canal. Es importante recordar el mapeo para interpretar los resultados.

## Aplicar Label Encoding a la columna 'canal_ingreso'

Ahora, aplicaremos Label Encoding a la columna `canal_ingreso`. Este método asignará un número entero único a cada categoría del canal de ingreso. Esto es útil para columnas con un orden implícito o cuando la cantidad de categorías es muy alta para One-Hot Encoding.

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Crear una copia del DataFrame para no modificar el original directamente
df_encoded_label = df_encoded.copy()

# Inicializar LabelEncoder
le = LabelEncoder()

# Aplicar Label Encoding a la columna 'canal_ingreso'
df_encoded_label['canal_ingreso_encoded'] = le.fit_transform(df_encoded_label['canal_ingreso'])

# Mostrar las primeras filas del DataFrame con la nueva columna codificada
print("DataFrame después de aplicar Label Encoding a 'canal_ingreso':")
display(df_encoded_label[['canal_ingreso', 'canal_ingreso_encoded']].head())

# Mostrar el mapeo de categorías a números
print("Mapeo de categorías de 'canal_ingreso' a números:")
for i, item in enumerate(le.classes_):
    print(f"{item} : {i}")

### Preparación para el Análisis Exploratorio

Antes de sumergirnos en el Análisis Exploratorio de Datos (EDA), verificamos la estructura del DataFrame después de las codificaciones y transformaciones realizadas. Esto nos asegura que las nuevas columnas están presentes y que los datos están en un formato adecuado para la visualización y el análisis.

In [ ]:
df_encoded_label.head(10)

## Análisis Exploratorio de Datos (EDA) Visual

El EDA es una fase crucial para entender las características de los datos. A través de visualizaciones, podemos identificar patrones, anomalías, distribuciones y relaciones entre las variables. Esta sección se enfoca en comprender cómo se distribuyen los tiempos de respuesta, los montos disputados, y cómo el estado actual de los reclamos se relaciona con otras variables categóricas.

## Exploratory Data Analysis (EDA)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Boxplot de Tiempo de Respuesta por Institución
plt.figure(figsize=(14, 8))
sns.boxplot(x='tiempo_respuesta_dias', y='institucion', data=df.sort_values('tiempo_respuesta_dias', ascending=False))
plt.title('Distribución del Tiempo de Respuesta por Institución')
plt.xlabel('Tiempo de Respuesta (Días)')
plt.ylabel('Institución')
plt.tight_layout()
plt.show()

# Boxplot de Tiempo de Respuesta por Producto
plt.figure(figsize=(14, 8))
sns.boxplot(x='tiempo_respuesta_dias', y='producto', data=df.sort_values('tiempo_respuesta_dias', ascending=False))
plt.title('Distribución del Tiempo de Respuesta por Producto')
plt.xlabel('Tiempo de Respuesta (Días)')
plt.ylabel('Producto')
plt.tight_layout()
plt.show()

# Boxplot de Tiempo de Respuesta por Estado Actual
plt.figure(figsize=(12, 7))
sns.boxplot(x='estado_actual', y='tiempo_respuesta_dias', data=df.sort_values('tiempo_respuesta_dias', ascending=False))
plt.title('Distribución del Tiempo de Respuesta por Estado Actual')
plt.xlabel('Estado Actual')
plt.ylabel('Tiempo de Respuesta (Días)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### Distribución del Tiempo de Respuesta y Monto Disputado

Los histogramas nos permiten visualizar la distribución de las variables numéricas clave: `tiempo_respuesta_dias` y `monto_disputado_clp`. Esto ayuda a entender si los datos siguen una distribución normal, si están sesgados, o si existen valores atípicos que puedan requerir un tratamiento especial.

## El boxplot nos permite identificar que no existen outlier en los datos

In [ ]:
# Plot distribution of 'tiempo_respuesta_dias'
plt.figure(figsize=(10, 6))
sns.histplot(df_encoded_label['tiempo_respuesta_dias'], bins=30, kde=True)
plt.title('Distribución de Tiempo de Respuesta (Días)')
plt.xlabel('Tiempo de Respuesta (Días)')
plt.ylabel('Frecuencia')
plt.tight_layout()
plt.show()


### Tendencia de Reclamos a lo Largo del Tiempo

Analizar el número de reclamos a lo largo del tiempo nos puede revelar estacionalidades, tendencias crecientes o decrecientes, o eventos específicos que pudieron haber causado picos en los reclamos. Para ello, convertimos la columna `fecha_ingreso` a formato de fecha y luego agrupamos por día para contar los reclamos.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Convert 'fecha_ingreso' to datetime objects
df_encoded_label['fecha_ingreso'] = pd.to_datetime(df_encoded_label['fecha_ingreso'])

# Set the aesthetic style of the plots
sns.set_style("whitegrid")

# Plot distribution of 'monto_disputado_clp'
plt.figure(figsize=(10, 6))
sns.histplot(df_encoded_label['monto_disputado_clp'], bins=50, kde=True)
plt.title('Distribución de Monto Disputado (CLP)')
plt.xlabel('Monto Disputado (CLP)')
plt.ylabel('Frecuencia')
plt.tight_layout()
plt.show()


### Distribución por Estado Actual

Visualizar la distribución de `estado_actual` nos da una idea de la proporción de reclamos que se resuelven, se cierran sin acuerdo, o están en proceso. Esta información es crucial para entender el rendimiento general del proceso de gestión de reclamos.

In [ ]:
# Complaints over time
complaints_over_time = df_encoded_label.groupby('fecha_ingreso').size().reset_index(name='count')

plt.figure(figsize=(12, 6))
sns.lineplot(x='fecha_ingreso', y='count', data=complaints_over_time)
plt.title('Número de Reclamos a lo Largo del Tiempo')
plt.xlabel('Fecha de Ingreso')
plt.ylabel('Número de Reclamos')
plt.tight_layout()
plt.show()


### Mapa de Calor de la Matriz de Correlación

Para entender las relaciones lineales entre las variables numéricas, calculamos y visualizamos la matriz de correlación. Un mapa de calor es una forma efectiva de identificar rápidamente qué variables están fuertemente correlacionadas (positiva o negativamente) entre sí.

In [ ]:
# Distribution of 'estado_actual'
plt.figure(figsize=(10, 6))
sns.countplot(y='estado_actual', data=df_encoded_label, order=df_encoded_label['estado_actual'].value_counts().index)
plt.title('Distribución de Reclamos por Estado Actual')
plt.xlabel('Conteo')
plt.ylabel('Estado Actual')
plt.tight_layout()
plt.show()


## Promedio de Tiempo de Respuesta por Institución

Para obtener una perspectiva de rendimiento de cada institución, calculamos el tiempo de respuesta promedio por banco y lo visualizamos. Esto permite identificar qué instituciones son más rápidas o lentas en gestionar los reclamos.

In [ ]:
# Seleccionar solo las columnas numéricas para el cálculo de la correlación
numerical_cols = df_encoded_label.select_dtypes(include=['number']).columns

# Calcular la matriz de correlación
corr_matrix = df_encoded_label[numerical_cols].corr()

# Configurar el tamaño de la figura
plt.figure(figsize=(12, 10))

# Crear un mapa de calor (heatmap) de la matriz de correlación
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
plt.title('Matriz de Correlación de Variables Numéricas')
plt.show()

### Visualización del Promedio de Tiempo de Respuesta por Institución

Un gráfico de barras es ideal para comparar el promedio de tiempo de respuesta entre las diferentes instituciones bancarias, facilitando la identificación de las que tienen un mejor o peor desempeño en este aspecto.

## Preparación de Datos para Regresión Lineal

Primero, aplicaremos One-Hot Encoding a las columnas categóricas restantes (`producto`, `motivo_principal`, `region_usuario`, `estado_actual`) que aún no han sido procesadas. Luego, definiremos nuestras características (X) y la variable objetivo (y).

### División de Datos para Regresión Lineal

Antes de entrenar el modelo, dividimos el DataFrame en características (X) y la variable objetivo (y), y luego en conjuntos de entrenamiento y prueba (`X_train`, `X_test`, `y_train`, `y_test`). Esto es crucial para evaluar el rendimiento del modelo en datos no vistos.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Realizar One-Hot Encoding para las columnas categóricas restantes
df_final_encoded = pd.get_dummies(df_encoded_label, columns=[
    'producto',
    'motivo_principal',
    'region_usuario',
    'estado_actual'
], drop_first=True) # drop_first=True para evitar la multicolinealidad

# La columna 'id_reclamo' no es una característica predictiva, la eliminamos
df_final_encoded = df_final_encoded.drop(columns=['id_reclamo', 'fecha_ingreso', 'institucion', 'canal_ingreso', 'producto', 'motivo_principal', 'region_usuario', 'estado_actual'], errors='ignore')

# Identificar y excluir columnas de 'estado_actual_' para evitar leakage temporal
estado_actual_cols_to_drop = [col for col in df_final_encoded.columns if col.startswith('estado_actual_')]

# Definir las características (X) y la variable objetivo (y)
# Excluimos 'tiempo_respuesta_dias' como variable objetivo y las columnas de 'estado_actual_'
X = df_final_encoded.drop(columns=['tiempo_respuesta_dias'] + estado_actual_cols_to_drop, axis=1, errors='ignore')
y = df_final_encoded['tiempo_respuesta_dias']

# Dividir los datos en conjuntos de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Dimensiones de X_train:", X_train.shape)
print("Dimensiones de X_test:", X_test.shape)
print("Columnas finales para X:", X.columns.tolist()[:5], "...")

### Creación de la Variable de Satisfacción del Cliente

Para el modelado de clasificación, derivamos una nueva variable binaria llamada `satisfaccion` a partir de la columna `estado_actual`. Un valor de `1` representa un reclamo 'Resuelto' (cliente satisfecho), y `0` representa cualquier otro estado (cliente no satisfecho).

## Creación de la Variable de Satisfacción del Cliente

Basándonos en el `estado_actual` de los reclamos, crearemos una variable binaria llamada `satisfaccion`.
Asumiremos que un reclamo con `estado_actual` de 'Resuelto' indica un cliente satisfecho (valor 1), mientras que cualquier otro estado (como 'Cerrado sin acuerdo', 'En proceso', 'Rechazado') indicará un cliente no satisfecho (valor 0).

In [ ]:
# Crear la variable binaria 'satisfaccion'
# 1 si el estado actual es 'Resuelto', 0 en caso contrario
df_final_encoded['satisfaccion'] = (df_final_encoded['estado_actual_Resuelto'] == True).astype(int)

print("Distribución de la variable 'satisfaccion':")
display(df_final_encoded['satisfaccion'].value_counts())

print("Primeras filas del DataFrame con la nueva columna 'satisfaccion':")
display(df_final_encoded[['estado_actual_Resuelto', 'satisfaccion']].head())

### Entrenamiento y Evaluación del Modelo de Regresión Lineal

Entrenamos un modelo de Regresión Lineal con los datos de entrenamiento y realizamos predicciones en el conjunto de prueba. Luego, evaluamos su rendimiento utilizando métricas como el Error Cuadrático Medio (MSE) y el Coeficiente de Determinación (R^2).

## Entrenamiento del Modelo de Regresión Lineal

Ahora, entrenaremos un modelo de regresión lineal con los datos preparados.

In [ ]:
# Inicializar y entrenar el modelo de Regresión Lineal
model = LinearRegression()
model.fit(X_train, y_train)

# Realizar predicciones en el conjunto de prueba
y_pred = model.predict(X_test)

# Evaluar el modelo
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Error Cuadrático Medio (MSE): {mse:.2f}")
print(f"Coeficiente de Determinación (R^2): {r2:.2f}")

### Coeficientes del Modelo de Regresión Lineal

Los coeficientes del modelo de Regresión Lineal nos indican la dirección y magnitud de la influencia de cada característica sobre la variable objetivo. Analizamos los coeficientes más influyentes para entender qué factores afectan más el `tiempo_respuesta_dias`.

## Visualización de Resultados

Para entender el rendimiento del modelo, visualizaremos las predicciones contra los valores reales. Esto nos ayudará a ver qué tan bien se ajusta el modelo a los datos.

También podemos examinar los coeficientes del modelo para entender la importancia de cada característica, pero con tantas características OHE, una visualización de los coeficientes principales podría ser más informativa que una lista completa.

In [ ]:
# Gráfico de Predicciones vs. Valores Reales
plt.figure(figsize=(10, 7))
sns.scatterplot(x=y_test, y=y_pred, alpha=0.6)
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', lw=2) # Línea de referencia y=x
plt.title('Predicciones del Modelo de Regresión Lineal vs. Valores Reales')
plt.xlabel('Tiempo de Respuesta Real (Días)')
plt.ylabel('Tiempo de Respuesta Predicho (Días)')
plt.grid(True)
plt.tight_layout()
plt.show()

### Análisis de Correlación Completa

Para complementar el análisis de los coeficientes, examinamos la correlación de todas las características codificadas con `tiempo_respuesta_dias`. Esto proporciona una visión más holística de las relaciones lineales en el conjunto de datos completo.

### Coeficientes del Modelo

Los coeficientes indican la influencia de cada característica en la variable objetivo. Un coeficiente positivo significa que a medida que la característica aumenta, la variable objetivo también tiende a aumentar (manteniendo otras características constantes). Un coeficiente negativo indica lo contrario.

In [ ]:
# Obtener coeficientes y sus nombres
coefficients = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': model.coef_
})

# Ordenar por valor absoluto del coeficiente para ver las más influyentes
coefficients['Abs_Coefficient'] = np.abs(coefficients['Coefficient'])
coefficients = coefficients.sort_values(by='Abs_Coefficient', ascending=False)

# Mostrar los 15 coeficientes más influyentes
print("Los 15 coeficientes más influyentes del modelo:")
display(coefficients.head(15))

### Entrenamiento y Evaluación del Árbol de Decisión

Para comparar con la Regresión Lineal, entrenamos un modelo de Árbol de Decisión para predecir `tiempo_respuesta_dias`. Evaluamos su rendimiento con MSE y R^2 para ver si un modelo no lineal se ajusta mejor a los datos.

## Correlación Completa de Variables (Incluyendo Características Codificadas)

Para obtener una visión más detallada de cómo todas las características (incluyendo las codificadas) se relacionan entre sí y con el `tiempo_respuesta_dias`, vamos a calcular y visualizar la matriz de correlación para el DataFrame `df_final_encoded`.

In [ ]:
# Calcular la matriz de correlación para el DataFrame final con todas las características codificadas
correlation_matrix_full = df_final_encoded.corr()

# Filtrar la correlación con la variable objetivo 'tiempo_respuesta_dias'
correlation_with_target = correlation_matrix_full['tiempo_respuesta_dias'].sort_values(ascending=False)

print("Correlación de todas las características con 'tiempo_respuesta_dias':")
display(correlation_with_target.head(15))
display(correlation_with_target.tail(15))

# Opcional: Visualizar un mapa de calor de la matriz de correlación completa (puede ser muy grande)
# plt.figure(figsize=(20, 18))
# sns.heatmap(correlation_matrix_full, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
# plt.title('Matriz de Correlación Completa de Variables')
# plt.show()


In [ ]:
average_response_time_by_institution = df.groupby('institucion')['tiempo_respuesta_dias'].mean().reset_index()

print("Promedio de tiempo de respuesta por institución:")
display(average_response_time_by_institution.sort_values(by='tiempo_respuesta_dias', ascending=False))

## Visualización del Promedio de Tiempo de Respuesta por Institución

In [ ]:
plt.figure(figsize=(12, 7))
sns.barplot(x='tiempo_respuesta_dias', y='institucion', data=average_response_time_by_institution.sort_values(by='tiempo_respuesta_dias', ascending=False), palette='viridis')
plt.title('Promedio de Tiempo de Respuesta por Institución')
plt.xlabel('Tiempo de Respuesta Promedio (Días)')
plt.ylabel('Institución')
plt.tight_layout()
plt.show()

### Preparación de Datos y Modelado: Regresión Lineal para `tiempo_respuesta_dias`

Nuestro primer objetivo de modelado es predecir el `tiempo_respuesta_dias` (tiempo de respuesta en días) utilizando un modelo de Regresión Lineal. Para ello, necesitamos preparar las características, incluyendo la aplicación de One-Hot Encoding a las columnas categóricas restantes y la división de los datos en conjuntos de entrenamiento y prueba.

## Creación y Entrenamiento de un Árbol de Decisión

Ahora, implementaremos un modelo de Árbol de Decisión para predecir el `tiempo_respuesta_dias`.

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Inicializar y entrenar el modelo de Árbol de Decisión
dt_model = DecisionTreeRegressor(random_state=42)
dt_model.fit(X_train, y_train)

# Realizar predicciones en el conjunto de prueba
y_pred_dt = dt_model.predict(X_test)

# Evaluar el modelo
mse_dt = mean_squared_error(y_test, y_pred_dt)
r2_dt = r2_score(y_test, y_pred_dt)

print(f"Error Cuadrático Medio (MSE) del Árbol de Decisión: {mse_dt:.2f}")
print(f"Coeficiente de Determinación (R^2) del Árbol de Decisión: {r2_dt:.2f}")

### Visualización de Predicciones del Árbol de Decisión

Similar a la Regresión Lineal, visualizamos las predicciones del Árbol de Decisión contra los valores reales para evaluar gráficamente su ajuste y detectar posibles problemas de sobreajuste o bajoajuste.

## Visualización de Resultados del Árbol de Decisión

Visualicemos las predicciones del modelo de Árbol de Decisión contra los valores reales.

In [ ]:
plt.figure(figsize=(10, 7))
sns.scatterplot(x=y_test, y=y_pred_dt, alpha=0.6)
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', lw=2) # Línea de referencia y=x
plt.title('Predicciones del Modelo de Árbol de Decisión vs. Valores Reales')
plt.xlabel('Tiempo de Respuesta Real (Días)')
plt.ylabel('Tiempo de Respuesta Predicho (Días)')
plt.grid(True)
plt.tight_layout()
plt.show()

### Exploración de la Distribución de Satisfacción

Iniciamos el análisis de la variable `satisfaccion` visualizando su distribución. Entender el balance de clases (satisfecho vs. no satisfecho) es crucial para el modelado de clasificación, ya que un desequilibrio severo puede afectar el rendimiento del modelo.

## Análisis de la Satisfacción del Cliente (Variable `satisfaccion`)

### Análisis Detallado de la Satisfacción del Cliente

A continuación, exploraremos la distribución de la variable de satisfacción que hemos creado y cómo varía la satisfacción de los clientes en función de diferentes categorías como la institución, el producto y el canal de ingreso. Los porcentajes en los gráficos representan la proporción de reclamos 'Resueltos' (clientes satisfechos) dentro de cada categoría.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Distribución de la variable 'satisfaccion'
plt.figure(figsize=(8, 6))
ax = sns.countplot(x='satisfaccion', data=df_final_encoded, palette='viridis')
plt.title('Distribución de la Satisfacción del Cliente (0: No Satisfecho, 1: Satisfecho)')
plt.xlabel('Satisfacción')
plt.ylabel('Número de Reclamos')
plt.xticks([0, 1], ['No Satisfecho', 'Satisfecho'])

# Añadir etiquetas de porcentaje
total = len(df_final_encoded['satisfaccion'])
for p in ax.patches:
    percentage = '{:.1f}%'.format(100 * p.get_height()/total)
    x = p.get_x() + p.get_width() / 2
    y = p.get_height()
    # Adjust offset to be relative to the plot height for better scaling
    ax.annotate(percentage, (x, y + ax.get_ylim()[1] * 0.02), ha='center', va='bottom', fontsize=10,
                color='white', bbox=dict(facecolor='black', alpha=0.8, edgecolor='none', boxstyle='round,pad=0.2'))

plt.tight_layout()
plt.show()

### Satisfacción por Institución, Producto y Canal de Ingreso

Analizamos cómo varía la satisfacción del cliente en función de diferentes variables categóricas. Estos gráficos de barras nos permiten identificar qué instituciones, productos o canales de ingreso están asociados con mayores o menores niveles de satisfacción.

In [ ]:
# Para analizar la satisfacción por institución, necesitamos revertir las columnas one-hot encoded de 'institucion' a una sola columna categórica
# Primero, obtengamos los nombres de las columnas 'institucion_' asegurándonos de que sean solo las OHE de tipo bool
institucion_cols = [col for col in df_final_encoded.columns if col.startswith('institucion_') and df_final_encoded[col].dtype == 'bool']

# Crear una nueva columna 'institucion_original' usando idxmax (índice del máximo valor, que será 1 para la columna activa)
df_final_encoded['institucion_original'] = df_final_encoded[institucion_cols].idxmax(axis=1).str.replace('institucion_', '')

# Calcular el porcentaje de satisfacción por institución
satisfaction_by_institution = df_final_encoded.groupby('institucion_original')['satisfaccion'].mean().reset_index()
satisfaction_by_institution['satisfaccion_porcentaje'] = satisfaction_by_institution['satisfaccion'] * 100

# 2. Satisfacción por Institución (como porcentaje)
plt.figure(figsize=(12, 7))
sns.barplot(x='satisfaccion_porcentaje', y='institucion_original', data=satisfaction_by_institution.sort_values(by='satisfaccion_porcentaje', ascending=False), palette='coolwarm')
plt.title('Porcentaje de Clientes Satisfechos por Institución')
plt.xlabel('Porcentaje de Satisfacción')
plt.ylabel('Institución')
plt.xlim(0, 110) # Establecer límites del eje X para acomodar las etiquetas con mayor margen

# Añadir etiquetas de porcentaje a las barras
for index, row in satisfaction_by_institution.sort_values(by='satisfaccion_porcentaje', ascending=False).iterrows():
    plt.text(row.satisfaccion_porcentaje + 3, index, f'{row.satisfaccion_porcentaje:.1f}%',
             color='white', ha="left", va="center", fontsize=10,
             bbox=dict(facecolor='black', alpha=0.8, edgecolor='none', boxstyle='round,pad=0.2'))

plt.tight_layout()
plt.show()

In [ ]:
# Para analizar la satisfacción por producto, necesitamos revertir las columnas one-hot encoded de 'producto' a una sola columna categórica
producto_cols = [col for col in df_final_encoded.columns if col.startswith('producto_') and df_final_encoded[col].dtype == 'bool']

# Crear una nueva columna 'producto_original' similar a 'institucion_original'
df_final_encoded['producto_original'] = df_final_encoded[producto_cols].idxmax(axis=1).str.replace('producto_', '')

# Calcular el porcentaje de satisfacción por producto
satisfaction_by_product = df_final_encoded.groupby('producto_original')['satisfaccion'].mean().reset_index()
satisfaction_by_product['satisfaccion_porcentaje'] = satisfaction_by_product['satisfaccion'] * 100

# 3. Satisfacción por Producto (como porcentaje)
plt.figure(figsize=(12, 7))
sns.barplot(x='satisfaccion_porcentaje', y='producto_original', data=satisfaction_by_product.sort_values(by='satisfaccion_porcentaje', ascending=False), palette='magma')
plt.title('Porcentaje de Clientes Satisfechos por Producto')
plt.xlabel('Porcentaje de Satisfacción')
plt.ylabel('Producto')
plt.xlim(0, 110) # Establecer límites del eje X para acomodar las etiquetas con mayor margen

# Añadir etiquetas de porcentaje a las barras
for index, row in satisfaction_by_product.sort_values(by='satisfaccion_porcentaje', ascending=False).iterrows():
    plt.text(row.satisfaccion_porcentaje + 3, index, f'{row.satisfaccion_porcentaje:.1f}%',
             color='white', ha="left", va="center", fontsize=10,
             bbox=dict(facecolor='black', alpha=0.8, edgecolor='none', boxstyle='round,pad=0.2'))

plt.tight_layout()
plt.show()

In [ ]:
# Para analizar la satisfacción por canal de ingreso, usaremos la columna 'canal_ingreso_encoded' del df_encoded_label
# Necesitamos el mapeo original para las etiquetas

# Volvemos al df_encoded_label que tiene el 'canal_ingreso' original y el 'canal_ingreso_encoded'
# Asumiendo que `le` (LabelEncoder) está aún disponible del paso anterior

# Crear un DataFrame temporal para la visualización con la columna 'satisfaccion'
# Asegurémonos de que 'df_final_encoded' y 'df_encoded_label' tengan el mismo índice
df_temp = df_encoded_label.copy()
df_temp['satisfaccion'] = df_final_encoded['satisfaccion'] # Asignar la columna 'satisfaccion'

# Calcular el porcentaje de satisfacción por canal de ingreso
satisfaction_by_channel = df_temp.groupby('canal_ingreso')['satisfaccion'].mean().reset_index()
satisfaction_by_channel['satisfaccion_porcentaje'] = satisfaction_by_channel['satisfaccion'] * 100

# 4. Satisfacción por Canal de Ingreso (como porcentaje)
plt.figure(figsize=(10, 6))
sns.barplot(x='satisfaccion_porcentaje', y='canal_ingreso', data=satisfaction_by_channel.sort_values(by='satisfaccion_porcentaje', ascending=False), palette='viridis')
plt.title('Porcentaje de Clientes Satisfechos por Canal de Ingreso')
plt.xlabel('Porcentaje de Satisfacción')
plt.ylabel('Canal de Ingreso')
plt.xlim(0, 110) # Establecer límites del eje X para acomodar las etiquetas con mayor margen

# Añadir etiquetas de porcentaje a las barras
for index, row in satisfaction_by_channel.sort_values(by='satisfaccion_porcentaje', ascending=False).iterrows():
    plt.text(row.satisfaccion_porcentaje + 3, index, f'{row.satisfaccion_porcentaje:.1f}%',
             color='white', ha="left", va="center", fontsize=10,
             bbox=dict(facecolor='black', alpha=0.8, edgecolor='none', boxstyle='round,pad=0.2'))

plt.tight_layout()
plt.show()

### Preparación de Datos para Modelos de Clasificación

Para los modelos de clasificación, necesitamos definir las características (`X_clf`) y la variable objetivo (`y_clf`, que es `satisfaccion`). Es importante excluir del conjunto de características cualquier columna que haya sido utilizada para derivar directamente `satisfaccion` para evitar la fuga de datos.

## Modelo de Clasificación: Predicción de Satisfacción del Cliente

Para predecir si un reclamo tiene alta probabilidad de ser resuelto (indicando satisfacción del cliente), construiremos un modelo de clasificación. La variable objetivo será `satisfaccion` (1 para resuelto, 0 para no resuelto).

**Asegúrese de haber ejecutado las celdas anteriores que crean y modifican `df_final_encoded` antes de continuar con esta sección.**

Primero, definiremos nuestras características (`X`) y la variable objetivo (`y`). Es crucial excluir del conjunto de características las columnas de `estado_actual` de las que se derivó directamente `satisfaccion`, así como las columnas originales no codificadas y `tiempo_respuesta_dias` (ya que fue la variable objetivo para la regresión anterior).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Definir la variable objetivo (y) para clasificación
y_clf = df_final_encoded['satisfaccion']

# Definir las características (X) para clasificación
# Excluir la columna 'satisfaccion' y las columnas originales que ya están codificadas
# También excluimos 'estado_actual_' columnas ya que 'satisfaccion' se deriva de ellas
# Y 'tiempo_respuesta_dias' porque fue la variable objetivo de la regresión anterior
X_clf = df_final_encoded.drop(columns=[
    'satisfaccion',
    'tiempo_respuesta_dias',
    'institucion_original',
    'producto_original',
    'estado_actual_Cerrado sin acuerdo',
    'estado_actual_En proceso',
    'estado_actual_Rechazado',
    'estado_actual_Resuelto' # Ya que 'satisfaccion' se deriva directamente de esta
], errors='ignore')

# Verificar las columnas finales de X_clf
print("Columnas finales de X_clf:", X_clf.columns.tolist())

# Dividir los datos en conjuntos de entrenamiento y prueba
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf)

print("\nDimensiones de X_train_clf:", X_train_clf.shape)
print("Dimensiones de X_test_clf:", X_test_clf.shape)
print("Distribución de clases en y_train_clf:\n", y_train_clf.value_counts(normalize=True))
print("Distribución de clases en y_test_clf:\n", y_test_clf.value_counts(normalize=True))

### Entrenamiento del Modelo de Regresión Logística

Entrenamos un modelo de Regresión Logística para predecir la `satisfaccion`. Este modelo es una buena línea base para problemas de clasificación binaria, y utilizamos `class_weight='balanced'` para mitigar el impacto del desequilibrio de clases.

## Entrenamiento del Modelo de Regresión Logística

Utilizaremos la Regresión Logística, un modelo lineal adecuado para tareas de clasificación binaria, para predecir la probabilidad de que un reclamo sea resuelto.

In [ ]:
# Inicializar y entrenar el modelo de Regresión Logística
# Usamos class_weight='balanced' para manejar el desequilibrio de clases
log_reg_model = LogisticRegression(random_state=42, solver='liblinear', max_iter=1000, class_weight='balanced')
log_reg_model.fit(X_train_clf, y_train_clf)

# Realizar predicciones en el conjunto de prueba
y_pred_clf = log_reg_model.predict(X_test_clf)
y_pred_proba_clf = log_reg_model.predict_proba(X_test_clf)[:, 1] # Probabilidad de ser 1 (satisfecho)

### Evaluación del Modelo de Regresión Logística

Evaluamos el rendimiento del modelo de Regresión Logística utilizando métricas de clasificación estándar como Accuracy, Precision, Recall y F1-Score, y visualizamos la matriz de confusión. Estas métricas nos ayudan a entender qué tan bien el modelo predice cada clase y dónde comete errores.

## Evaluación del Modelo de Clasificación

Evaluaremos el rendimiento del modelo utilizando métricas clave para clasificación, como la exactitud (accuracy), precisión, exhaustividad (recall) y la puntuación F1, además de la matriz de confusión.

In [ ]:
print(f"Accuracy: {accuracy_score(y_test_clf, y_pred_clf):.2f}")
print(f"Precision: {precision_score(y_test_clf, y_pred_clf):.2f}")
print(f"Recall: {recall_score(y_test_clf, y_pred_clf):.2f}")
print(f"F1-Score: {f1_score(y_test_clf, y_pred_clf):.2f}")

print("\nClassification Report:")
print(classification_report(y_test_clf, y_pred_clf))

# Matriz de Confusión
cm = confusion_matrix(y_test_clf, y_pred_clf)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['No Satisfecho (0)', 'Satisfecho (1)'],
            yticklabels=['No Satisfecho (0)', 'Satisfecho (1)'])
plt.xlabel('Predicción')
plt.ylabel('Valor Real')
plt.title('Matriz de Confusión')
plt.show()

### Coeficientes del Modelo de Regresión Logística para Satisfacción

Los coeficientes de la Regresión Logística nos muestran la dirección y la fuerza con la que cada característica influye en la probabilidad de que un reclamo sea 'Satisfecho'. Los coeficientes con mayor valor absoluto son los más influyentes.

## Coeficientes del Modelo de Regresión Logística

Examinemos los coeficientes del modelo para entender la importancia relativa de cada característica en la predicción de la satisfacción del cliente. Un coeficiente positivo alto indica que la característica aumenta la probabilidad de satisfacción, mientras que un coeficiente negativo alto la disminuye.

In [ ]:
import pandas as pd
import numpy as np

coefficients_clf = pd.DataFrame({
    'Feature': X_clf.columns,
    'Coefficient': log_reg_model.coef_[0]
})

coefficients_clf['Abs_Coefficient'] = np.abs(coefficients_clf['Coefficient'])
coefficients_clf = coefficients_clf.sort_values(by='Abs_Coefficient', ascending=False)

print("Los 20 coeficientes más influyentes del modelo de Regresión Logística para Satisfacción:")
display(coefficients_clf.head(20))

### Entrenamiento del Modelo de Random Forest

Considerando las limitaciones de la Regresión Logística, probamos un modelo de Random Forest. Este modelo de conjunto suele ofrecer mayor precisión y manejar mejor las relaciones no lineales y el desequilibrio de clases, especialmente con la configuración `class_weight='balanced'`.

## Modelo de Clasificación: Random Forest

Dado que la Regresión Logística mostró limitaciones, especialmente con el desequilibrio de clases, probaremos un modelo de Random Forest. Los Random Forests son ensembles de árboles de decisión que a menudo proporcionan mayor precisión y manejan bien las relaciones no lineales y el desequilibrio de clases si se configura correctamente.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Inicializar y entrenar el modelo de Random Forest
# Usamos class_weight='balanced' para manejar el desequilibrio de clases
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced', n_jobs=-1)
rf_model.fit(X_train_clf, y_train_clf)

# Realizar predicciones en el conjunto de prueba
y_pred_rf = rf_model.predict(X_test_clf)
y_pred_proba_rf = rf_model.predict_proba(X_test_clf)[:, 1] # Probabilidad de ser 1 (satisfecho)

### Evaluación del Modelo de Random Forest

Evaluamos el rendimiento del Random Forest con las mismas métricas de clasificación y la matriz de confusión. Comparamos estos resultados con los de la Regresión Logística para determinar qué modelo tiene un mejor desempeño general y en la detección de la clase minoritaria.

## Evaluación del Modelo de Random Forest

Evaluaremos el rendimiento del modelo de Random Forest utilizando las mismas métricas clave de clasificación, así como la matriz de confusión.

In [ ]:
print(f"Accuracy (Random Forest): {accuracy_score(y_test_clf, y_pred_rf):.2f}")
print(f"Precision (Random Forest): {precision_score(y_test_clf, y_pred_rf):.2f}")
print(f"Recall (Random Forest): {recall_score(y_test_clf, y_pred_rf):.2f}")
print(f"F1-Score (Random Forest): {f1_score(y_test_clf, y_pred_rf):.2f}")

print("\nClassification Report (Random Forest):")
print(classification_report(y_test_clf, y_pred_rf))

# Matriz de Confusión
cm_rf = confusion_matrix(y_test_clf, y_pred_rf)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['No Satisfecho (0)', 'Satisfecho (1)'],
            yticklabels=['No Satisfecho (0)', 'Satisfecho (1)'])
plt.xlabel('Predicción')
plt.ylabel('Valor Real')
plt.title('Matriz de Confusión (Random Forest)')
plt.show()

### Importancia de las Características del Random Forest

Una de las ventajas del Random Forest es su capacidad para proporcionar una medida de la importancia de cada característica. Esto nos ayuda a identificar los factores más relevantes que el modelo utiliza para predecir la satisfacción del cliente.

## Importancia de las Características (Feature Importances) del Random Forest

Los modelos de Random Forest pueden proporcionar una medida de la importancia de cada característica en el proceso de toma de decisiones del modelo. Esto nos ayudará a entender qué factores son más relevantes para predecir la satisfacción del cliente.

In [ ]:
importances = rf_model.feature_importances_
feature_names = X_clf.columns

forest_importances = pd.Series(importances, index=feature_names)

# Seleccionar las 20 características más importantes
top_20_features = forest_importances.nlargest(20)

plt.figure(figsize=(12, 8))
sns.barplot(x=top_20_features.values, y=top_20_features.index, palette='viridis')
plt.title('Top 20 Importancia de Características (Random Forest)')
plt.xlabel('Importancia')
plt.ylabel('Característica')
plt.tight_layout()
plt.show()

print("Las 20 características más influyentes del modelo Random Forest para Satisfacción:")
display(top_20_features.to_frame(name='Importancia'))

### Aplicación de SMOTE para Balancear Clases

Para abordar el desequilibrio de clases de manera más efectiva, aplicamos SMOTE (Synthetic Minority Over-sampling Technique) al conjunto de entrenamiento. SMOTE genera ejemplos sintéticos de la clase minoritaria, lo que ayuda al modelo a aprender patrones más diversos y a mejorar la detección de la clase 'No Satisfecho'.

### Manejo del Desbalance de Clases con SMOTE

**SMOTE (Synthetic Minority Over-sampling Technique)** es una técnica de sobremuestreo que genera ejemplos sintéticos de la clase minoritaria. A diferencia de simplemente duplicar ejemplos existentes, SMOTE crea nuevas instancias que son combinaciones de varios ejemplos de la clase minoritaria, lo que ayuda al modelo a aprender patrones más diversos y a mejorar su rendimiento en la detección de la clase minoritaria.

Esto es crucial porque, aunque `class_weight='balanced'` en Random Forest ajusta los pesos de las clases durante el entrenamiento, a veces no es suficiente cuando el desbalance es muy severo, o cuando el algoritmo subyacente se beneficia de ver más ejemplos de la clase minoritaria durante el proceso de división de nodos. Al aplicar SMOTE, esperamos que el modelo tenga más ejemplos para aprender sobre la clase 'No Satisfecho' (0), lo que debería mejorar métricas como el F1-Score para esa clase.

In [ ]:
# Instalar imbalanced-learn si no está instalado
!pip install imbalanced-learn

from imblearn.over_sampling import SMOTE

# Inicializar SMOTE
smote = SMOTE(random_state=42)

# Aplicar SMOTE al conjunto de entrenamiento
X_train_res, y_train_res = smote.fit_resample(X_train_clf, y_train_clf)

print("Dimensiones de X_train_clf antes de SMOTE:", X_train_clf.shape)
print("Distribución de clases en y_train_clf antes de SMOTE:\n", y_train_clf.value_counts(normalize=True))

print("\nDimensiones de X_train_res después de SMOTE:", X_train_res.shape)
print("Distribución de clases en y_train_res después de SMOTE:\n", y_train_res.value_counts(normalize=True))

### Reentrenamiento y Evaluación del Random Forest con SMOTE

Después de balancear el conjunto de entrenamiento con SMOTE, reentrenamos el modelo de Random Forest. Evaluamos nuevamente su rendimiento para ver si la aplicación de SMOTE ha mejorado la capacidad del modelo para identificar la clase minoritaria ('No Satisfecho') sin un sacrificio significativo en el rendimiento general.

### Reentrenamiento del Modelo Random Forest con Datos Balanceados

Ahora que hemos balanceado el conjunto de entrenamiento usando SMOTE, procederemos a reentrenar el modelo Random Forest. Esta vez, el modelo aprenderá de un número equitativo de ejemplos de ambas clases, lo que debería llevar a una mejor identificación de la clase minoritaria (`No Satisfecho`).

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Inicializar y entrenar el modelo de Random Forest con los datos resampleados
# class_weight='balanced' no es estrictamente necesario aquí después de SMOTE,
# pero no perjudica y puede actuar como una capa extra de protección.
rf_model_smote = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced', n_jobs=-1)
rf_model_smote.fit(X_train_res, y_train_res)

# Realizar predicciones en el conjunto de prueba (no resampleado)
y_pred_rf_smote = rf_model_smote.predict(X_test_clf)


### Importancia de las Características del Random Forest (con SMOTE)

Volvemos a analizar la importancia de las características después de reentrenar el Random Forest con SMOTE. Esto nos permite ver si el balanceo de clases ha modificado la relevancia percibida de ciertos factores en la predicción de la satisfacción.

### Resumen de Importancia de Características del Random Forest (con SMOTE)

El análisis de la importancia de las características del modelo Random Forest reentrenado con SMOTE nos proporciona una visión clave sobre qué factores influyen más en la probabilidad de que un reclamo sea resuelto (satisfacción del cliente). Los factores más influyentes, en orden descendente de importancia, son:

*   **Monto Disputado (CLP):** Es, por mucho, el factor más determinante, representando aproximadamente el **41.1%** de la importancia total. Esto sugiere que el valor monetario del reclamo es crucial en el desenlace de la satisfacción del cliente.
*   **Canal de Ingreso (Codificado):** El canal a través del cual se presenta el reclamo es el segundo factor más importante, con un **8.6%** de relevancia. Esto indica que la plataforma o método de interacción inicial juega un rol significativo.
*   **Producto - Cuenta Vista / Cuenta RUT:** Este tipo de producto específico es el tercer factor más influyente, con un **2.2%** de importancia. La naturaleza del producto reclamado tiene un peso considerable.
*   **Producto - Tarjeta de Crédito:** Muy cerca del anterior, este producto también es un factor relevante, con **2.18%** de importancia.
*   **Institución - Banco Estado:** La institución involucrada, en este caso Banco Estado, figura con un **2.18%** de importancia, destacando que la entidad bancaria también afecta la satisfacción.
*   **Otros Factores:** Le siguen con menor, pero notable, importancia otros productos (`producto_Crédito de Consumo`, `producto_Cuenta Corriente`), instituciones (`institucion_Banco de Chile`, `institucion_Scotiabank Chile`, `institucion_Banco Santander Chile`, `institucion_Itaú Chile`, `institucion_Banco de Crédito e Inversiones (BCI)`) y motivos principales de reclamo.

En general, el **monto disputado** es el conductor principal de la predicción de satisfacción, seguido por el **canal de ingreso** y el **tipo de producto/institución**. Esta información es vital para que las instituciones bancarias puedan enfocar sus esfuerzos en mejorar la satisfacción del cliente de manera más estratégica.

### Evaluación del Modelo Random Forest Reentrenado con SMOTE

Evaluaremos el rendimiento del modelo Random Forest después de aplicar SMOTE, prestando especial atención al F1-Score de la clase 'No Satisfecho' para verificar si la técnica ha mejorado la detección de esta clase.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

print(f"Accuracy (Random Forest con SMOTE): {accuracy_score(y_test_clf, y_pred_rf_smote):.2f}")
print(f"Precision (Random Forest con SMOTE): {precision_score(y_test_clf, y_pred_rf_smote):.2f}")
print(f"Recall (Random Forest con SMOTE): {recall_score(y_test_clf, y_pred_rf_smote):.2f}")
print(f"F1-Score (Random Forest con SMOTE): {f1_score(y_test_clf, y_pred_rf_smote):.2f}")

print("\nClassification Report (Random Forest con SMOTE):")
print(classification_report(y_test_clf, y_pred_rf_smote))

# Matriz de Confusión
cm_rf_smote = confusion_matrix(y_test_clf, y_pred_rf_smote)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_rf_smote, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['No Satisfecho (0)', 'Satisfecho (1)'],
            yticklabels=['No Satisfecho (0)', 'Satisfecho (1)'])
plt.xlabel('Predicción')
plt.ylabel('Valor Real')
plt.title('Matriz de Confusión (Random Forest con SMOTE)')
plt.show()

In [ ]:
importances_smote = rf_model_smote.feature_importances_
feature_names_smote = X_clf.columns

forest_importances_smote = pd.Series(importances_smote, index=feature_names_smote)

# Seleccionar las 20 características más importantes
top_20_features_smote = forest_importances_smote.nlargest(20)

plt.figure(figsize=(12, 8))
sns.barplot(x=top_20_features_smote.values, y=top_20_features_smote.index, palette='viridis')
plt.title('Top 20 Importancia de Características (Random Forest con SMOTE)')
plt.xlabel('Importancia')
plt.ylabel('Característica')
plt.tight_layout()
plt.show()

print("Las 20 características más influyentes del modelo Random Forest (con SMOTE) para Satisfacción:")
display(top_20_features_smote.to_frame(name='Importancia'))

**F1-Score del modelo Random Forest con SMOTE:**


Para la clase 'No Satisfecho' (0): el F1-Score ha mejorado de 0.27 a 0.32.
Para la clase 'Satisfecho' (1): el F1-Score ha disminuido ligeramente de 0.70 a 0.68.
Análisis de las métricas:

Accuracy: Disminuyó de 0.58 a 0.56.
Precision (Clase 0): Se mantuvo en 0.34.
Recall (Clase 0): Mejoró de 0.23 a 0.29.
El F1-Score de la clase minoritaria ('No Satisfecho') ha mostrado una modesta mejora. Esto indica que SMOTE ha ayudado al modelo a ser un poco más eficaz en la identificación de reclamos no satisfechos, lo cual era nuestro objetivo principal. Sin embargo, esta mejora se ha logrado a expensas de una ligera disminución en el rendimiento de la clase mayoritaria y en la precisión general del modelo. Es un trade-off común cuando se balancean clases.

**La matriz de confusión también lo refleja:**

Predicción No Satisfecho (0)	Predicción Satisfecho (1)
Real No Satisfecho (0)	810 (Verdaderos Negativos)	1965 (Falsos Positivos)
Real Satisfecho (1)	1543 (Falsos Negativos)	3682 (Verdaderos Positivos)
Ahora identificamos más 'No Satisfechos' correctamente (810 vs 643 antes), pero también clasificamos erróneamente más 'No Satisfechos' como 'Satisfechos' (1965 vs 2132 antes) y más 'Satisfechos' como 'No Satisfechos' (1543 vs 1259 antes).

En resumen, SMOTE ha logrado mover un poco el foco del modelo hacia la clase minoritaria, haciendo que sea ligeramente mejor en detectarla, aunque aún queda espacio para mejorar. Si el objetivo es maximizar la detección de la clase 'No Satisfecho', esta es una mejora positiva. Si tu prioridad es la precisión general, podríamos explorar otras técnicas o ajustar los hiperparámetros.

## Explicabilidad del Modelo con SHAP (SHapley Additive exPlanations)

Para entender cómo las características individuales impactan las predicciones de nuestro modelo Random Forest, utilizaremos SHAP. SHAP es una librería que nos ayuda a explicar la salida de cualquier modelo de machine learning, mostrando la contribución de cada característica a la predicción para una instancia individual o para el modelo global.

In [ ]:
# Instalar la librería SHAP si no está instalada
!pip install shap

In [ ]:
import shap
import matplotlib.pyplot as plt
import numpy as np

# Para modelos basados en árboles como Random Forest, TreeExplainer es eficiente
# Es recomendable usar una muestra del conjunto de entrenamiento como background dataset para TreeExplainer

# Convertir columnas booleanas a enteros (0 y 1) en X_train_res explícitamente
# Esto es necesario porque SHAP TreeExplainer puede tener problemas con dtypes booleano 'O'
for col in X_train_res.select_dtypes(include=['bool']).columns:
    X_train_res[col] = X_train_res[col].astype(int)

# Muestrear el conjunto de entrenamiento para el background dataset de SHAP
# Se recomienda usar un subconjunto más pequeño para grandes datasets para ahorrar memoria y tiempo
# Por ejemplo, 100 o 1000 muestras, dependiendo de los recursos disponibles
background_data = X_train_res.sample(n=1000, random_state=42) # Ajustar 'n' si hay problemas de memoria

# Calcular los valores SHAP para una muestra del conjunto de prueba
# Usamos X_test_clf para evaluar el rendimiento del modelo en datos no vistos
# Tomamos una muestra para visualizaciones para evitar sobrecarga de memoria
sample_X_test_clf = X_test_clf.sample(n=1000, random_state=42) # Ajustar 'n' si hay problemas de memoria

# Convertir columnas booleanas a enteros (0 y 1) en sample_X_test_clf explícitamente
for col in sample_X_test_clf.select_dtypes(include=['bool']).columns:
    sample_X_test_clf[col] = sample_X_test_clf[col].astype(int)

# FIX: Crear un masker explícito y establecer max_samples para usar todos los datos de fondo.
# La advertencia sugiere que un masker interno se está creando con max_samples=100,
# lo que causa que solo 100 de los 1000 samples de background_data sean usados.
# Al crear un masker.Independent explícito, podemos controlar este parámetro.
masker = shap.maskers.Independent(background_data, max_samples=background_data.shape[0])

# Inicializar TreeExplainer con el masker explícito.
# También especificamos feature_perturbation='interventional' para asegurar que el masker sea usado de esta manera,
# ya que la advertencia proviene de un proceso de muestreo similar a interventional SHAP.
explainer = shap.TreeExplainer(rf_model_smote, data=masker, feature_perturbation='interventional')

shap_values = explainer.shap_values(sample_X_test_clf)

print("SHAP values calculados.")

### Gráfico de Resumen (Summary Plot) de SHAP

El gráfico de resumen de SHAP muestra la importancia de las características a nivel global y cómo cada característica afecta la predicción (si aumenta o disminuye la salida del modelo). Cada punto representa una instancia de predicción:

*   **Eje Y:** Características ordenadas por su importancia.
*   **Eje X:** Valor SHAP, que indica el impacto de la característica en la predicción.
*   **Color:** Representa el valor de la característica (rojo = valor alto, azul = valor bajo).

Este gráfico nos permite ver qué características son las más influyentes y la dirección general de su impacto.

In [ ]:
# Para modelos de clasificación, shap_values es una lista de arrays, uno por clase.
# Para una predicción binaria, tomamos los valores SHAP para la clase positiva (satisfacción = 1).
# Si shap_values es un array 3D (muestras, características, clases), necesitamos rebanar la última dimensión.
shap.summary_plot(shap_values[:, :, 1], sample_X_test_clf, plot_type="dot", show=False)
plt.title('SHAP Summary Plot para la Satisfacción del Cliente (Clase 1: Satisfecho)')
plt.tight_layout()
plt.show()

### Gráfico de Dependencia (Dependence Plot) de SHAP

Un gráfico de dependencia de SHAP muestra el efecto de una característica sobre la predicción del modelo en todas las instancias del conjunto de datos. También puede mostrar interacciones con otra característica. Aquí visualizaremos las dos características más importantes identificadas en el summary plot.

Vamos a generar gráficos de dependencia para:
1.  `monto_disputado_clp` (la característica más influyente)
2.  `canal_ingreso_encoded` (la segunda característica más influyente)

In [ ]:
# Gráfico de Dependencia para 'monto_disputado_clp'
# Usamos shap_values[:, :, 1] para obtener los SHAP values de la clase positiva.
shap.dependence_plot("monto_disputado_clp", shap_values[:, :, 1], sample_X_test_clf, show=False)
plt.title('SHAP Dependence Plot: Monto Disputado (CLP)')
plt.tight_layout()
plt.show()

# Gráfico de Dependencia para 'canal_ingreso_encoded'
# Podemos ver la interacción con la siguiente característica más importante o con la que tenga más sentido.
# Por simplicidad, lo haremos solo para 'canal_ingreso_encoded'.
# Usamos shap_values[:, :, 1] para obtener los SHAP values de la clase positiva.
shap.dependence_plot("canal_ingreso_encoded", shap_values[:, :, 1], sample_X_test_clf, show=False)
plt.title('SHAP Dependence Plot: Canal de Ingreso (Codificado)')
plt.tight_layout()
plt.show()

# Opcional: Gráfico de fuerza para una instancia específica (para ver una explicación individual)
# instance_to_explain = sample_X_test_clf.iloc[0]
# shap.initjs()
# shap.force_plot(explainer.expected_value[1], shap_values[:, :, 1][0,:], instance_to_explain)

print("Gráficos de dependencia generados para las características principales.")

## Conclusiones de Negocio y Recomendaciones para el Sector Bancario

Basándonos en el análisis de los modelos de Regresión Logística y Random Forest, así como en la interpretabilidad proporcionada por SHAP, podemos extraer varias conclusiones y formular recomendaciones clave para mejorar la satisfacción del cliente en el sector bancario.

## Análisis Comparativo de Modelos y Justificación de Resultados

Hemos explorado tres enfoques de modelado para predecir la satisfacción del cliente (`satisfaccion`):

1.  **Regresión Logística con `class_weight='balanced'`**
2.  **Random Forest con `class_weight='balanced'`**
3.  **Random Forest con SMOTE y `class_weight='balanced'`**

Examinemos los resultados clave:

### 1. Regresión Logística
*   **Accuracy:** 0.36
*   **Precision (Clase 0 'No Satisfecho'):** 0.35
*   **Recall (Clase 0 'No Satisfecho'):** 0.97
*   **F1-Score (Clase 0 'No Satisfecho'):** 0.51
*   **F1-Score (Clase 1 'Satisfecho'):** 0.07

**Análisis:** Este modelo mostró un **Recall muy alto para la clase 'No Satisfecho'**, lo que significa que identificó a casi todos los clientes insatisfechos. Sin embargo, su **precisión fue muy baja**, lo que indica que clasificó erróneamente a muchos clientes satisfechos como insatisfechos. El F1-Score general fue pobre, y el F1-Score para la clase 'Satisfecho' fue extremadamente bajo, lo que sugiere que el modelo no aprendió bien a diferenciar entre las clases y tendió a predecir casi todo como 'No Satisfecho' debido al `class_weight='balanced'` y la naturaleza lineal del modelo frente a la complejidad de los datos.

### 2. Random Forest (sin SMOTE)
*   **Accuracy:** 0.58
*   **Precision (Clase 0 'No Satisfecho'):** 0.34
*   **Recall (Clase 0 'No Satisfecho'):** 0.23
*   **F1-Score (Clase 0 'No Satisfecho'):** 0.27
*   **F1-Score (Clase 1 'Satisfecho'):** 0.70

**Análisis:** El Random Forest mejoró significativamente el rendimiento general y el F1-Score de la clase mayoritaria ('Satisfecho'). Sin embargo, para nuestra clase minoritaria de interés ('No Satisfecho'), el F1-Score fue de **0.27**, lo que, aunque no es terrible, indica que el modelo aún tiene dificultades para identificar correctamente esta clase. El Recall para 'No Satisfecho' fue de solo 0.23, lo que significa que el modelo se perdió la mayoría de los casos de clientes insatisfechos.

### 3. Random Forest (con SMOTE)
*   **Accuracy:** 0.56
*   **Precision (Clase 0 'No Satisfecho'):** 0.34
*   **Recall (Clase 0 'No Satisfecho'):** 0.29
*   **F1-Score (Clase 0 'No Satisfecho'):** 0.32
*   **F1-Score (Clase 1 'Satisfecho'):** 0.68

**Análisis:** Al aplicar SMOTE y reentrenar el Random Forest, observamos una **mejora en el F1-Score de la clase minoritaria ('No Satisfecho'), pasando de 0.27 a 0.32**. También se notó un aumento en el Recall para esta clase (de 0.23 a 0.29). Esta mejora, aunque modesta, confirma que SMOTE ayudó al modelo a aprender mejor los patrones de la clase minoritaria. El costo fue una ligera disminución en la precisión general y el F1-Score de la clase mayoritaria, lo cual es un compromiso aceptable si el objetivo principal es mejorar la detección de la clase minoritaria.

### Justificación de la Elección del Modelo

Basándonos en el objetivo de **mejorar la identificación de la clase minoritaria ('No Satisfecho')**, el **Random Forest entrenado con datos balanceados por SMOTE** es el modelo preferido. Aunque la mejora en el F1-Score de la clase minoritaria no fue drástica, es la mejor opción entre los modelos evaluados en términos de balance entre precisión y recall para esta clase, sin comprometer excesivamente el rendimiento general del modelo.

La Regresión Logística, a pesar de su alto Recall para la clase 0, carecía de precisión y de un buen rendimiento para la clase 1, haciéndolo poco útil. El Random Forest sin SMOTE fue mejor en general, pero aún débil para la clase minoritaria. SMOTE nos permitió dar un paso adelante en la dirección correcta para abordar el desequilibrio de clases y el rendimiento de la clase 'No Satisfecho'.

**Próximos pasos:** Aunque hemos logrado una mejora, todavía hay margen. Se podría explorar la **optimización de hiperparámetros** del Random Forest (por ejemplo, `max_depth`, `min_samples_leaf`) o considerar **otras técnicas de balanceo** de clases o algoritmos de clasificación más avanzados para intentar mejorar aún más el F1-Score de la clase 'No Satisfecho' sin sacrificar demasiado el rendimiento general.

### 1. Factores Clave que Influyen en la Satisfacción (Según Importancia de Características del Random Forest con SMOTE):

La importancia de las características del Random Forest (reentrenado con SMOTE) reveló que ciertas variables tienen un impacto significativo en la probabilidad de que un reclamo sea resuelto (satisfacción). Los factores más influyentes son:

*   **Monto Disputado (CLP):** Es el factor más crítico con una importancia del 41%. Valores altos de este monto tienden a disminuir la satisfacción, mientras que montos menores están asociados a una mayor probabilidad de resolución. Esto sugiere que el manejo de reclamos de alto valor podría requerir protocolos más robustos o atención especializada.
*   **Canal de Ingreso:** Con una importancia del 8.6%, el canal a través del cual se presenta el reclamo también es relevante. El 'canal_ingreso_encoded' muestra una influencia en la satisfacción, aunque no podemos inferir la dirección exacta sin un análisis más profundo de cómo se correlacionan los canales específicos con la satisfacción.
*   **Producto y la Institución:** Diversos tipos de productos (como 'Cuenta Vista / Cuenta RUT' y 'Tarjeta de Crédito') y algunas instituciones bancarias (como 'Banco Estado') también aparecen entre las características más importantes, indicando que el tipo de producto o la institución específica pueden influir en la satisfacción del cliente.

### 2. Rendimiento del Modelo de Clasificación (Random Forest con SMOTE):

El modelo de Random Forest con SMOTE superó a la Regresión Logística, ofreciendo una mejor capacidad predictiva de la satisfacción del cliente, especialmente para la clase minoritaria de 'No Satisfecho'.

*   Con un **Recall** del **70%** para la clase 'Satisfecho' (clase 1), el modelo de Random Forest es razonablemente bueno identificando reclamos que probablemente serán resueltos. Su **Precisión** del **65%** para esta clase sugiere que todavía hay margen para reducir los falsos positivos (reclamos predichos como resueltos que no lo fueron). Para la clase 'No Satisfecho' (clase 0), el **F1-Score mejoró a 0.32** (con un Recall de 29% y Precision de 34%), lo que indica una mejor capacidad para detectar estos casos críticos, aunque aún con espacio para optimización.

### 3. Recomendaciones de Negocio:

*   **Priorización de Reclamos por Monto Disputado:** Desarrollar un sistema para priorizar reclamos con montos disputados elevados, ya que estos tienen una mayor probabilidad de resultar en insatisfacción. Asignar recursos especializados para la gestión de estos casos.
*   **Optimización por Canal de Atención y Producto:** Realizar un análisis detallado de la satisfacción por cada canal de ingreso y tipo de producto. Si ciertos canales o productos muestran consistentemente baja satisfacción, invertir en capacitación del personal de soporte, mejorar los procesos internos, o ajustar la comunicación para esos segmentos específicos.
*   **Programas de Retención Proactiva:** Utilizar las predicciones del modelo (especialmente para la clase 'No Satisfecho') para identificar proactivamente a los clientes en riesgo de insatisfacción. Ofrecer soluciones personalizadas o programas de retención antes de que la insatisfacción escale.
*   **Revisión de Políticas Internas:** Si los motivos de reclamo o productos específicos (como 'Cuenta Vista / Cuenta RUT' o 'Tarjeta de Crédito') aparecen constantemente como importantes en la insatisfacción, revisar las políticas o procesos asociados para identificar y corregir las causas raíz.
*   **Monitoreo y Mejora Continua del Modelo:** Implementar un monitoreo constante del rendimiento del modelo en producción. Recalibrar y reentrenar el modelo periódicamente con nuevos datos para asegurar su relevancia y precisión a lo largo del tiempo, y explorar la optimización de hiperparámetros o algoritmos más avanzados para mejorar aún más el F1-Score de la clase 'No Satisfecho'.

No se pudo utilizar la libreria shap por resuros se utilizaron otras tecnicas para tratar de replicar sus resultados